# Cleaning
We clean the messages of train_v2 and test_v2.
We look only at the train data to decide the rules.
Then we use the same rules on the test data.

## 1. Load the data

In [1]:
import re
import pandas as pd
from collections import Counter
train = pd.read_csv('../data/train_v2.csv')
train['label'].value_counts()

label
ham     3601
spam     501
Name: count, dtype: int64

## 2. All the symbols in the messages
We count every character that is not a letter, a number or a space.

In [2]:
all_text = ' '.join(train['text'])
symbols = Counter(re.findall(r'[^a-zA-Z0-9\s]', all_text))
print('Different symbols:', len(symbols))
symbols.most_common()

Different symbols: 50


[('.', 8075),
 ("'", 1370),
 (',', 1370),
 ('?', 1171),
 ('!', 998),
 ('&', 625),
 (';', 525),
 (':', 504),
 ('-', 393),
 (')', 345),
 ('/', 279),
 ('*', 251),
 ('"', 227),
 ('£', 212),
 ('#', 204),
 ('ü', 98),
 ('(', 88),
 ('+', 81),
 ('Ü', 43),
 ('\x92', 34),
 ('=', 30),
 ('‘', 29),
 ('@', 28),
 ('$', 20),
 ('>', 19),
 ('|', 18),
 ('…', 15),
 ('%', 9),
 ('–', 7),
 ('_', 6),
 ('<', 5),
 ('’', 3),
 ('é', 3),
 ('“', 2),
 ('~', 2),
 ('\x94', 2),
 ('\\', 2),
 ('[', 2),
 (']', 2),
 ('\x96', 2),
 ('¡', 2),
 ('è', 1),
 ('\x91', 1),
 ('ú', 1),
 ('É', 1),
 ('»', 1),
 ('\x93', 1),
 ('鈥', 1),
 ('┾', 1),
 ('〨', 1)]

## 3. What is different between ham and spam?
For each thing, we count how many ham and spam messages have it.
We also show the percent of each class.

In [3]:
ham = train['label'] == 'ham'
spam = train['label'] == 'spam'
checks = [('£', '£'), ('$', r'\$'), ('hidden info', '&lt;'), ('&amp;', '&amp;'),
          ('website', r'(?i)www\.|http|\w\.(com|co\.uk|net|org|biz|tv|ac|info)\b'),
          ('T&C', r"(?i)\bt'?s?\s?(&|and)\s?c'?s?\b|\bT Cs\b"), ('ü or Ü', '[üÜ]'),
          ('5+ digits', r'\d{5,}'), ('phone with spaces', r'\b0\d{3,4}[ -]\d{3,4}[ -]\d{3,4}\b'),
          ('pence 150p', r'(?i)\b\d+(\.\d+)?p\b'), ('happy face', r'[:;]-?[)DPp]'),
          ('sad face', r':-?\('), ('ALL CAPS word', r'\b[A-Z]{2,}\b'), ('!', '!'), ('...', r'\.\.\.')]
for name, pattern in checks:
    has = train['text'].str.contains(pattern)
    print(name, '| ham', has[ham].sum(), round(100 * has[ham].mean(), 1),
          '| spam', has[spam].sum(), round(100 * has[spam].mean(), 1))

£ | ham 5 0.1 | spam 166 33.1
$ | ham 12 0.3 | spam 4 0.8
hidden info | ham 164 4.6 | spam 0 0.0
&amp; | ham 41 1.1 | spam 0 0.0
website | ham 7 0.2 | spam 92 18.4
T&C | ham 0 0.0 | spam 53 10.6
ü or Ü | ham 111 3.1 | spam 0 0.0
5+ digits | ham 3 0.1 | spam 378 75.4
phone with spaces | ham 0 0.0 | spam 13 2.6
pence 150p | ham 0 0.0 | spam 101 20.2
happy face | ham 242 6.7 | spam 0 0.0
sad face | ham 26 0.7 | spam 0 0.0
ALL CAPS word | ham 249 6.9 | spam 363 72.5
! | ham 422 11.7 | spam 233 46.5
... | ham 574 15.9 | spam 6 1.2


C:\Users\Amjad\AppData\Local\Temp\ipykernel_14692\3145653991.py:10: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has = train['text'].str.contains(pattern)


## 4. Style of the message
Cleaning will remove capital letters, "!" and "...".
But they are different between ham and spam.
So we save them as numbers in new columns before cleaning:
- length: number of characters
- caps: number of words written in capital letters
- excl: number of "!"
- dots: number of "..."

In [4]:
train['length'] = train['text'].str.len()
train['caps'] = train['text'].str.count(r'\b[A-Z]{2,}\b')
train['excl'] = train['text'].str.count('!')
train['dots'] = train['text'].str.count(r'\.\.\.')
train.groupby('label')[['length', 'caps', 'excl', 'dots']].mean().round(2)

,length,caps,excl,dots
label,,,,
ham,71.04,0.40,0.18,0.27
spam,136.42,1.93,0.70,0.01


## 5. Cleaning function
The rules, in order:
1. Faces like :) :D :P become "happyface", and :( becomes "sadface".
2. All letters become small letters.
3. Hidden info like `&lt;#&gt;` becomes "hiddeninfo".
4. Other codes like `&amp;` are removed.
5. Websites become "url".
6. Money with £ becomes "pounds", and money with $ becomes "dollars".
7. Prices like 150p become "pence".
8. Phone numbers and numbers with 5 or more digits become "longnumber".
9. Apostrophes are removed, so "don't" becomes "dont".
10. All forms of T&C become "tandc".
11. Any other symbol becomes a space. We keep the letter ü.
12. We split the message into words using the spaces.

In [5]:
def clean(text):
    text = re.sub(r'[:;]-?\)', ' happyface ', text)
    text = re.sub(r':-?[DPp]\b', ' happyface ', text)
    text = re.sub(r':-?\(', ' sadface ', text)
    text = text.lower()
    text = re.sub(r'&lt;[^&]*&gt;', ' hiddeninfo ', text)
    text = re.sub(r'&lt;\S*', ' hiddeninfo ', text)
    text = re.sub(r'&(gt|amp);', ' ', text)
    text = re.sub(r'(http\S*|www\S*|\S+\.(com|co\.uk|net|org|biz|tv|ac|info)\S*)', ' url ', text)
    text = re.sub(r'£\s?\d[\d,]*(\.\d+)?', ' pounds ', text)
    text = re.sub(r'\$\s?\d[\d,]*(\.\d+)?', ' dollars ', text)
    text = re.sub(r'\b\d+(\.\d+)?p\b', ' pence ', text)
    text = re.sub(r'\b\d+ppm\b', ' pence ', text)
    text = re.sub(r'\b0\d{3,4}[ -]\d{3,4}[ -]\d{3,4}\b', ' longnumber ', text)
    text = re.sub(r'\d{5,}', ' longnumber ', text)
    for ch in ["'", '‘', '’', '\x91', '\x92']:
        text = text.replace(ch, '')
    text = re.sub(r'\bts?\s?&\s?cs?', ' tandc ', text)
    text = re.sub(r'\bts?\s(and\s)?cs\b', ' tandc ', text)
    text = re.sub(r'[^a-z0-9ü ]', ' ', text)
    return ' '.join(text.split())

## 6. Test the rules
We use the function on the train data.
Then we count how many ham and spam messages have each new word.

In [6]:
train['clean_text'] = train['text'].apply(clean)
for w in ['happyface', 'sadface', 'hiddeninfo', 'tandc', 'url',
          'pounds', 'dollars', 'pence', 'longnumber', 'ü']:
    has = train['clean_text'].str.contains(r'\b' + w + r'\b')
    print(w, '| ham', has[ham].sum(), '| spam', has[spam].sum())

happyface | ham 230 | spam 0
sadface | ham 26 | spam 0
hiddeninfo | ham 163 | spam 0
tandc | ham 0 | spam 56
url | ham 11 | spam 93
pounds | ham 7 | spam 176
dollars | ham 8 | spam 3
pence | ham 0 | spam 129
longnumber | ham 3 | spam 382
ü | ham 109 | spam 0


## 7. Clean train and test, then save
We use the same function on both files.
The function has fixed rules, so it does not learn anything from the test data.

In [7]:
test = pd.read_csv('../data/test_v2.csv')
for df in [train, test]:
    df['length'] = df['text'].str.len()
    df['caps'] = df['text'].str.count(r'\b[A-Z]{2,}\b')
    df['excl'] = df['text'].str.count('!')
    df['dots'] = df['text'].str.count(r'\.\.\.')
    df['clean_text'] = df['text'].apply(clean)
print('Empty in train:', (train['clean_text'] == '').sum())
print('Empty in test:', (test['clean_text'] == '').sum())
train.to_csv('../data/train_clean_v2.csv', index=False)
test.to_csv('../data/test_clean_v2.csv', index=False)
print(train.shape, test.shape)

Empty in train: 0
Empty in test: 0
(4102, 7) (1026, 7)


## 8. Checks
- Is the same clean text in train and test?
- Is the same clean text repeated in train?
- Does the same clean text have two labels?
- How many different words before and after cleaning?

In [8]:
overlap = set(train['clean_text']) & set(test['clean_text'])
print('Same clean text in train and test:', len(overlap))
print('Repeated clean texts in train:', train['clean_text'].duplicated().sum())
conflict = train.groupby('clean_text')['label'].nunique().gt(1).sum()
print('Same clean text, different labels:', conflict)
raw_words = set(' '.join(train['text']).lower().split())
clean_words = set(' '.join(train['clean_text']).split())
print('Words before:', len(raw_words), '| after:', len(clean_words))
for t in overlap:
    print(t[:100])

Same clean text in train and test: 4
Repeated clean texts in train: 19
Same clean text, different labels: 0
Words before: 11802 | after: 7303
urgent your mobile number has been awarded with a pounds prize guaranteed call longnumber from land 
congratulations thanks to a good friend u have won the pounds xmas prize 2 claim is easy just call l
do you want a new nokia 3510i colour phone delivered tomorrow with 200 free minutes to any mobile 10
hot live fantasies call now longnumber just pence per min ntt ltd po box 1327 croydon cr9 5wb 0870 i


## 9. Before and after
One example for each rule.

In [9]:
for w in ['happyface', 'sadface', 'hiddeninfo', 'tandc', 'url',
          'pounds', 'pence', 'longnumber', 'ü']:
    i = train[train['clean_text'].str.contains(r'\b' + w + r'\b')].index[0]
    print('RULE  :', w)
    print('BEFORE:', train['text'][i])
    print('AFTER :', train['clean_text'][i])
    print()

RULE  : happyface
BEFORE: I'm working technical support :)voice process.
AFTER : im working technical support happyface voice process

RULE  : sadface
BEFORE: Boo what time u get out? U were supposed to take me shopping today. :(
AFTER : boo what time u get out u were supposed to take me shopping today sadface

RULE  : hiddeninfo
BEFORE:  &lt;DECIMAL&gt; m but its not a common car here so its better to buy from china or asia. Or if i find it less expensive. I.ll holla
AFTER : hiddeninfo m but its not a common car here so its better to buy from china or asia or if i find it less expensive i ll holla

RULE  : tandc
BEFORE: Please call Amanda with regard to renewing or upgrading your current T-Mobile handset free of charge. Offer ends today. Tel 0845 021 3680 subject to T's and C's
AFTER : please call amanda with regard to renewing or upgrading your current t mobile handset free of charge offer ends today tel longnumber subject to tandc

RULE  : url
BEFORE: Hello from Orange. For 1 month'